<img src="https://raw.githubusercontent.com/AliAkrami1375/dcho/main/assets/logo.png" width="90"># dcho — training on ColabPersian text-to-speech, trained here and checkpointed to the Hugging Face Hub.**Read this before running.** Colab sessions end — on the free tier afterabout 12 hours, sooner if the tab closes or the runtime is reclaimed. Thisnotebook is built around that fact rather than against it: it pushes acheckpoint to the Hub every few thousand steps and, on the next run, picksup exactly where it stopped. Losing a session costs the minutes since thelast checkpoint, not the run.**Before you start:** set `Runtime → Change runtime type → GPU`, and addyour Hugging Face token as a Colab secret named `HF_TOKEN` (the key icon inthe left sidebar). The token needs write access.

## 1 · What hardware did we get?

In [ ]:
import subprocess, torchprint(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],                     capture_output=True, text=True).stdout.strip() or "NO GPU")if not torch.cuda.is_available():    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then rerun.")# Rough throughput relative to an L4, used to project how long a phase will# take on whatever Colab handed us this session.name = torch.cuda.get_device_name()RELATIVE = {"T4": 0.37, "L4": 1.00, "A100": 2.50, "V100": 0.75, "P100": 0.45}speed = next((v for k, v in RELATIVE.items() if k in name), 0.6)print(f"{name}  ~{speed:.2f}x an L4")

## 2 · Install and fetch the code

In [ ]:
%%capture!pip -q install soundfile huggingface_hub>=0.28!git clone -q https://github.com/AliAkrami1375/dcho.git /content/dcho || (cd /content/dcho && git pull -q)

In [ ]:
import sys, ossys.path.insert(0, "/content/dcho")os.chdir("/content/dcho")from google.colab import userdataos.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")from huggingface_hub import HfApiprint("signed in as", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])!python -m unittest discover -s tests -p "test_model.py" 2>&1 | tail -3

## 3 · Configuration`micro` is the cheap probe — use it to check that a configuration changebehaves, not to judge audio. `base` is the real model.

In [ ]:
CONFIG   = "micro"                       # micro | nano | baseDATASET  = "DibaAi/dcho-packed"          # packed by jobs/pack_subset.py
TIER     = "tier_a"                      # tier_a 65.3h/~4.6GB · tier_b 180.8h/~13GBRUN_REPO = "DibaAi/dcho-run-micro"       # checkpoints land hereMAX_STEPS = 30_000CHECKPOINT_EVERY = 2_000                 # keep this small; sessions dieimport jsoncfg = json.load(open(f"configs/{CONFIG}.json"))steps_per_sec_l4 = {"micro": 4.0, "nano": 2.2, "base": 1.5}[CONFIG]eta = MAX_STEPS / (steps_per_sec_l4 * speed) / 3600print(f"{CONFIG}: {MAX_STEPS} steps, roughly {eta:.1f} h on this GPU")if eta > 11:    print(f"  longer than a free session — expect about {eta/11:.0f} runs of this notebook")

## 4 · DataThe packed dataset holds only the clips this phase trains on, as FLAC withthe transcript and speaker vector attached. Downloading it once per sessionis far cheaper than streaming the full 47 GB corpus repeatedly.

In [ ]:
from huggingface_hub import snapshot_downloadimport timet0 = time.time()DATA_DIR = snapshot_download(DATASET, repo_type="dataset",                             local_dir="/content/data", token=os.environ["HF_TOKEN"])print(f"downloaded in {(time.time()-t0)/60:.1f} min")!du -sh /content/data && df -h /content | tail -1

## 5 · ResumeThe first thing the notebook does is ask the Hub whether this run alreadyhas a checkpoint. If it does, training continues from it; if not, it startsfresh. Nothing else in the notebook needs to know which case it is.

In [ ]:
from huggingface_hub import HfApi, hf_hub_downloadapi = HfApi(token=os.environ["HF_TOKEN"])api.create_repo(RUN_REPO, repo_type="model", private=True, exist_ok=True)resume_from, start_step = None, 0try:    files = api.list_repo_files(RUN_REPO)    ckpts = sorted(f for f in files if f.startswith("step_") and f.endswith(".pth"))    if ckpts:        latest = max(ckpts, key=lambda f: int(f[5:-4]))        start_step = int(latest[5:-4])        resume_from = hf_hub_download(RUN_REPO, latest, token=os.environ["HF_TOKEN"])        print(f"resuming from {latest}  ({start_step} steps already done)")except Exception as e:    print("no prior checkpoint:", e)if resume_from is None:    print("starting from scratch")

## 6 · Build the model

In [ ]:
import torchfrom dcho.model.synthesizer import Synthesizerfrom dcho.text.phonemes import N_SYMBOLSfrom dcho.train.trainer import Trainermodel_cfg = {k: v for k, v in cfg["model"].items() if not k.startswith("_")}net = Synthesizer(    n_vocab=N_SYMBOLS,    spec_channels=cfg["data"]["filter_length"] // 2 + 1,    segment_size=cfg["train"]["segment_size"] // cfg["data"]["hop_length"],    n_speakers=0,    speaker_embed_dim=192,          # continuous ECAPA vector    **model_cfg,)print(f"{net.n_parameters()/1e6:.2f}M parameters "      f"({net.n_parameters(inference_only=True)/1e6:.2f}M at inference)")# Colab bills by the hour, so the guard is set from the session rather than# from a dollar figure.cfg["train"]["max_cost_usd"] = Nonetrainer = Trainer(cfg, net, output_dir="/content/out", device="cuda", speaker_embed_dim=192)if resume_from:    trainer.load(resume_from)    print("resumed at step", trainer.state.step)

## 7 · Data loader

In [ ]:
from torch.utils.data import DataLoaderfrom dcho.data.dataset import DataConfig, PackedSpeechDataset, collatedata_cfg = DataConfig(**{k: v for k, v in cfg["data"].items()                         if k in DataConfig.__dataclass_fields__})ds = PackedSpeechDataset(f"/content/data/{TIER}", data_cfg)loader = DataLoader(ds, batch_size=cfg["train"]["batch_size"], num_workers=2,                    collate_fn=lambda b: collate(b, data_cfg),                    pin_memory=True, persistent_workers=True)batch = next(iter(loader))print({k: tuple(v.shape) for k, v in batch.items()})

## 8 · TrainEvery `CHECKPOINT_EVERY` steps the weights go to the Hub. If the sessiondies, rerun the notebook from the top and it will resume.

In [ ]:
from dcho.train.guards import TrainingHaltedimport tracebackdef push(tag):    path = trainer.save(tag)    api.upload_file(path_or_fileobj=str(path), path_in_repo=f"{tag}.pth",                    repo_id=RUN_REPO, repo_type="model",                    commit_message=f"{tag} — step {trainer.state.step}")    print(f"  pushed {tag}")def on_log(m):    if m["step"] % CHECKPOINT_EVERY == 0:        push(f"step_{m['step']}")try:    trainer.train(loader, max_steps=MAX_STEPS, checkpoint_every=10**9, on_log=on_log)    push("final")except TrainingHalted as halt:    print(f"HALTED — {halt.reason}: {halt.detail}")    push("halted")except KeyboardInterrupt:    print("interrupted; saving so the session is not wasted")    push(f"step_{trainer.state.step}")except Exception:    traceback.print_exc()    push(f"step_{trainer.state.step}")

## Push the checkpoint back

Run this **whether the run finished, halted or was interrupted** — a halt
writes a checkpoint precisely so it can be kept. Without this cell the
session ending costs everything since the last push, which is the failure
the whole resume mechanism exists to prevent.


In [ ]:
from pathlib import Path

out = Path(trainer.output_dir)
ckpts = sorted(out.glob("*.pth"))
print("local checkpoints:", [c.name for c in ckpts])

api.upload_folder(
    folder_path=str(out), repo_id=RUN_REPO, repo_type="model",
    allow_patterns=["*.pth", "summary.json"],
    commit_message=f"{CONFIG} @ step {trainer.state.step}",
)
print(f"pushed to {RUN_REPO} at step {trainer.state.step}")
print("spend this session:", trainer.budget.report())


## 9 · ListenAlignment has to lock in before anything sounds like speech. Before roughlystep 10,000 expect noise — that is normal, and `align_H` in the training logfalling towards zero is the signal that it is working.

In [ ]:
import numpy as npfrom IPython.display import Audio, displayfrom dcho.text.frontend import Frontendfe = Frontend()net.eval()spk = torch.from_numpy(np.frombuffer(batch["speaker_vector_raw"][0], dtype=np.float16)                       .astype(np.float32)).unsqueeze(0).cuda() \      if "speaker_vector_raw" in batch else batch["sid"][:1].cuda()for text in ["سلام، حال شما چطور است؟", "زبان فارسی یکی از زبان‌های کهن جهان است."]:    ids = torch.tensor([fe(text).ids]).cuda()    with torch.no_grad():        audio, *_ = net.infer(ids, torch.tensor([ids.shape[1]]).cuda(), sid=spk)    print(text)    display(Audio(audio.squeeze().cpu().numpy(), rate=cfg["data"]["sampling_rate"]))

---### If the session diesRerun from cell 1. It downloads the data again (a few minutes) and resumesfrom the last checkpoint on the Hub.### Keeping a free session aliveLeave the tab open and visible. Colab reclaims idle runtimes, and abackgrounded tab counts as idle. Colab Pro adds background execution, whichis the only reliable way to run for more than a few hours unattended.### Watching from outsideEvery checkpoint is on the Hub under `RUN_REPO`, so progress is visiblewithout touching this notebook.

---
## What this costs

Colab sells compute units at $9.99 per 100. Set `hourly_rate_usd` in the
config to (units per hour x 0.0999) so `BudgetGuard` stops at a real
ceiling rather than a guessed one.

| GPU | units/hour | ~ $/hour | same GPU on HF Jobs |
|---|---|---|---|
| T4 | ~1.19 | **~0.12** | 0.40 |
| A100 40 GB | ~5.40 | **~0.54** | 2.50 |

Cheaper per GPU-hour than Jobs, and the tradeoff is operational rather
than financial: a Job is submitted and forgotten, Colab wants a browser
session and takes the machine back when that session ends.

**Start with `micro`.** About 30k steps, and it answers the only questions
worth asking before spending real money: does alignment converge, and do
the loss curves look healthy. It says nothing about audio quality — that
is what it is for.
